## 2. Preprocesado de los datos unidos

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
processed_input = Path("../data/processed/merged_data.parquet")
preprocessed_output = Path("../data/processed/preprocessed_data.parquet")

# Add src/ to Python path
sys.path.append(str(Path("../src").resolve()))
from data.downcast import downcast_numeric_columns, get_non_downcastable_columns


# Display options for large datasets (we have 3.4M rows!!)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [2]:
df = pd.read_parquet(processed_input, engine="fastparquet")  # same engine as saved (fastparquet)
print("Original shape:", df.shape)
df.head()

Original shape: (3476825, 81)


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,source_file
0,0,0,2018-02-14 08:31:01,112641719,3,0,0,0,0,0,0.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.026633,5.632086e+07,139.300036,56320958,56320761,112641719,5.632086e+07,139.300036,56320958,56320761,0,0.000000,0.000000,0,0,0,0,0,0,0,0,0.026633,0.000000,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.00,0.000000,0.000000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0.0,0.0,0,0,56320859.5,139.300036,56320958,56320761,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter
1,0,0,2018-02-14 08:33:50,112641466,3,0,0,0,0,0,0.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.026633,5.632073e+07,114.551299,56320814,56320652,112641466,5.632073e+07,114.551299,56320814,56320652,0,0.000000,0.000000,0,0,0,0,0,0,0,0,0.026633,0.000000,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.00,0.000000,0.000000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0.0,0.0,0,0,56320733.0,114.551299,56320814,56320652,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter
2,0,0,2018-02-14 08:36:39,112638623,3,0,0,0,0,0,0.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.026634,5.631931e+07,301.934596,56319525,56319098,112638623,5.631931e+07,301.934596,56319525,56319098,0,0.000000,0.000000,0,0,0,0,0,0,0,0,0.026634,0.000000,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.00,0.000000,0.000000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0.0,0.0,0,0,56319311.5,301.934596,56319525,56319098,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter
3,22,6,2018-02-14 08:40:13,6453966,15,10,1239,2273,744,0,82.600000,196.741237,976,0,227.300000,371.677892,544.161528,3.873587,2.689152e+05,247443.778966,673900,22,6453966,4.609976e+05,123109.423588,673900,229740,5637902,626433.555556,455082.214224,1167293,554,0,0,0,0,488,328,2.324152,1.549435,0,976,135.076923,277.834760,77192.153846,0,0,0,1,0,0,0,0,0,140.48,82.600000,227.300000,0,0,0,0,0,0,15,1239,10,2273,65535,233,6,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter
4,22,6,2018-02-14 08:40:23,8804066,14,11,1143,2209,744,0,81.642857,203.745545,976,0,200.818182,362.249864,380.733175,2.839597,3.668361e+05,511356.609733,1928102,21,8804066,6.772358e+05,532416.970959,1928102,246924,7715481,771548.100000,755543.082717,2174893,90,0,0,0,0,456,360,1.590174,1.249423,0,976,128.923077,279.763032,78267.353846,0,0,0,1,0,0,0,0,0,134.08,81.642857,200.818182,0,0,0,0,0,0,14,1143,11,2209,5808,233,6,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter


#### 1) Tratamiento de duplicados

In [3]:
# Total number of duplicate rows
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns identical): {num_duplicates}")

Total duplicate rows (all columns identical): 245679


In [4]:
# Show first 5 duplicate rows
df[df.duplicated()].head()

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,source_file
96,21,6,2018-02-14 10:33:26,3,1,1,0,0,0,0,0.0,0.0,0,0,0.0,0.0,0.0,666666.666667,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,40,20,333333.333333,333333.333333,0,0,0.0,0.0,0.0,0,0,0,1,0,0,0,0,1,0.0,0.0,0.0,0,0,0,0,0,0,1,0,1,0,26883,0,0,40,0.0,0.0,0,0,0.0,0.0,0,0,FTP-BruteForce,Wednesday-14-02-2018_TrafficForML_CICFlowMeter
98,21,6,2018-02-14 10:33:26,2,1,1,0,0,0,0,0.0,0.0,0,0,0.0,0.0,0.0,1000000.000000,2.0,0.0,2,2,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,40,20,500000.000000,500000.000000,0,0,0.0,0.0,0.0,0,0,0,1,0,0,0,0,1,0.0,0.0,0.0,0,0,0,0,0,0,1,0,1,0,26883,0,0,40,0.0,0.0,0,0,0.0,0.0,0,0,FTP-BruteForce,Wednesday-14-02-2018_TrafficForML_CICFlowMeter
99,21,6,2018-02-14 10:33:26,2,1,1,0,0,0,0,0.0,0.0,0,0,0.0,0.0,0.0,1000000.000000,2.0,0.0,2,2,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,40,20,500000.000000,500000.000000,0,0,0.0,0.0,0.0,0,0,0,1,0,0,0,0,1,0.0,0.0,0.0,0,0,0,0,0,0,1,0,1,0,26883,0,0,40,0.0,0.0,0,0,0.0,0.0,0,0,FTP-BruteForce,Wednesday-14-02-2018_TrafficForML_CICFlowMeter
100,21,6,2018-02-14 10:33:26,2,1,1,0,0,0,0,0.0,0.0,0,0,0.0,0.0,0.0,1000000.000000,2.0,0.0,2,2,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,40,20,500000.000000,500000.000000,0,0,0.0,0.0,0.0,0,0,0,1,0,0,0,0,1,0.0,0.0,0.0,0,0,0,0,0,0,1,0,1,0,26883,0,0,40,0.0,0.0,0,0,0.0,0.0,0,0,FTP-BruteForce,Wednesday-14-02-2018_TrafficForML_CICFlowMeter
101,21,6,2018-02-14 10:33:26,2,1,1,0,0,0,0,0.0,0.0,0,0,0.0,0.0,0.0,1000000.000000,2.0,0.0,2,2,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,40,20,500000.000000,500000.000000,0,0,0.0,0.0,0.0,0,0,0,1,0,0,0,0,1,0.0,0.0,0.0,0,0,0,0,0,0,1,0,1,0,26883,0,0,40,0.0,0.0,0,0,0.0,0.0,0,0,FTP-BruteForce,Wednesday-14-02-2018_TrafficForML_CICFlowMeter


Aunque haya muchos duplicados la mayoría de ellos corresponden a ataques de bruteforce por lo cual eliminarlos nos afectaría negativamente a la posterior clasificación.

In [5]:
df[df.duplicated()]['Label'].value_counts()

Label
FTP-BruteForce           154008
SSH-Bruteforce            70267
DDOS attack-HOIC          17551
Benign                     3072
DoS attacks-Slowloris       705
DoS attacks-GoldenEye        53
Infilteration                23
Name: count, dtype: int64

La cantidad de las etiquetas en total es:

In [6]:
df['Label'].value_counts()

Label
Benign                   2262573
DDOS attack-HOIC          686012
FTP-BruteForce            193360
SSH-Bruteforce            187589
Infilteration              93063
DoS attacks-GoldenEye      41508
DoS attacks-Slowloris      10990
DDOS attack-LOIC-UDP        1730
Name: count, dtype: int64

#### 2) Tratamiento de NAs

En este caso todos se concentran en el Flow de Byts por tanto podríamos poner un valor de 0.0 como tinen algunos o directamente eliminar las filas relacionadas.

In [7]:
df.isna().sum()[df.isna().sum()!=0]

Flow Byts/s    9032
dtype: int64

Como podemos ver otra vez estos pueden estar relacionados a algunos ataques pero no es común, por tanto podemos suponer que es algun error de trafico normal. Y la proporción de los ataques que sufren de esto es muy baja comparada con el total.

In [8]:
df[df.isna().any(axis=1)]['Label'].value_counts()

Label
Benign            8597
Infilteration      429
FTP-BruteForce       6
Name: count, dtype: int64

Considero que poner un valor de 0.0 alteraría la información y por tanto voy a decidirme por borrar dichas filas.

In [9]:
df=df.dropna()
df.isna().sum()

Dst Port         0
Protocol         0
Timestamp        0
Flow Duration    0
Tot Fwd Pkts     0
                ..
Idle Std         0
Idle Max         0
Idle Min         0
Label            0
source_file      0
Length: 81, dtype: int64

#### 3) Tipos de datos

Ahora debo ajustar los tipos de datos a sus correspondientes.

In [10]:
df.dtypes.value_counts()

int64             54
float64           24
object             2
datetime64[us]     1
Name: count, dtype: int64

In [11]:
print(df.dtypes)

Dst Port                  int64
Protocol                  int64
Timestamp        datetime64[us]
Flow Duration             int64
Tot Fwd Pkts              int64
                      ...      
Idle Std                float64
Idle Max                  int64
Idle Min                  int64
Label                    object
source_file              object
Length: 81, dtype: object


Cambiamos a category los que se pueden categorizar, protocolos, etiquetas y csv de origen.

In [12]:
df['Protocol'] = df['Protocol'].astype('category')
df['Label'] = df['Label'].astype('category')
df['source_file'] = df['source_file'].astype('category')
print(df.dtypes)

Dst Port                  int64
Protocol               category
Timestamp        datetime64[us]
Flow Duration             int64
Tot Fwd Pkts              int64
                      ...      
Idle Std                float64
Idle Max                  int64
Idle Min                  int64
Label                  category
source_file            category
Length: 81, dtype: object


Y otro problema es el tipo de datos de Timestamp, que utilizaré para crear una nueva columna una para los segundos que han pasado desde medianoche (00:00:00). No cojo la del día porque los datasets que tomo son de miercoles y jueves y no va a ser relevante, cada uno es de un dia de esos. Esto nos va a permitir mayor eficiencia y aplicabilidad al analisis con coste de interpretabilidad pero para eso conservaremos la columna de Timestamp que contiene toda la info.

In [13]:
ts = df['Timestamp']
df['time_seconds'] = (
    ts.dt.hour * 3600 +
    ts.dt.minute * 60 +
    ts.dt.second
).astype('int32')
df.head()

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,source_file,time_seconds
0,0,0,2018-02-14 08:31:01,112641719,3,0,0,0,0,0,0.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.026633,5.632086e+07,139.300036,56320958,56320761,112641719,5.632086e+07,139.300036,56320958,56320761,0,0.000000,0.000000,0,0,0,0,0,0,0,0,0.026633,0.000000,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.00,0.000000,0.000000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0.0,0.0,0,0,56320859.5,139.300036,56320958,56320761,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter,30661
1,0,0,2018-02-14 08:33:50,112641466,3,0,0,0,0,0,0.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.026633,5.632073e+07,114.551299,56320814,56320652,112641466,5.632073e+07,114.551299,56320814,56320652,0,0.000000,0.000000,0,0,0,0,0,0,0,0,0.026633,0.000000,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.00,0.000000,0.000000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0.0,0.0,0,0,56320733.0,114.551299,56320814,56320652,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter,30830
2,0,0,2018-02-14 08:36:39,112638623,3,0,0,0,0,0,0.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.026634,5.631931e+07,301.934596,56319525,56319098,112638623,5.631931e+07,301.934596,56319525,56319098,0,0.000000,0.000000,0,0,0,0,0,0,0,0,0.026634,0.000000,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.00,0.000000,0.000000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0.0,0.0,0,0,56319311.5,301.934596,56319525,56319098,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter,30999
3,22,6,2018-02-14 08:40:13,6453966,15,10,1239,2273,744,0,82.600000,196.741237,976,0,227.300000,371.677892,544.161528,3.873587,2.689152e+05,247443.778966,673900,22,6453966,4.609976e+05,123109.423588,673900,229740,5637902,626433.555556,455082.214224,1167293,554,0,0,0,0,488,328,2.324152,1.549435,0,976,135.076923,277.834760,77192.153846,0,0,0,1,0,0,0,0,0,140.48,82.600000,227.300000,0,0,0,0,0,0,15,1239,10,2273,65535,233,6,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter,31213
4,22,6,2018-02-14 08:40:23,8804066,14,11,1143,2209,744,0,81.642857,203.745545,976,0,200.818182,362.249864,380.733175,2.839597,3.668361e+05,511356.609733,1928102,21,8804066,6.772358e+05,532416.970959,1928102,246924,7715481,771548.100000,755543.082717,2174893,90,0,0,0,0,456,360,1.590174,1.249423,0,976,128.923077,279.763032,78267.353846,0,0,0,1,0,0,0,0,0,134.08,81.642857,200.818182,0,0,0,0,0,0,14,1143,11,2209,5808,233,6,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter,31223


Finalmente un punto importante será la conversión del tipo de datos pasar de 64 a 32 porque no nos hace falta tanto, pero primero comprobamos los casos en los que no se puede hacer este downcast. Una vez lo hemos verificado procedemos a hacer downcast en las posibles esto va a conllevar un mejor rendimiento general del dataset.

In [14]:
downcast_numeric_columns(df)

Columnas enteras que NO se pueden downcast:
['Flow Duration', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Max', 'Fwd IAT Min', 'Idle Max', 'Idle Min']

Columnas float que NO se pueden downcast:
['Flow Byts/s', 'Flow Pkts/s']

Downcasting completado.


In [15]:
df.dtypes.value_counts()

uint32            43
float32           22
int64              8
int32              3
float64            2
category           1
datetime64[us]     1
category           1
category           1
Name: count, dtype: int64

#### 4) Extras

Vemos que todo va acorde con la teoría de la red, protocolos y puertos existentes.

In [16]:
df['Protocol'].value_counts()

Protocol
6     2762525
17     667611
0       37657
Name: count, dtype: int64

In [17]:
print('Intervalo de puertos: [',df['Dst Port'].min(), ', ', df['Dst Port'].max(), ']')

Intervalo de puertos: [ 0 ,  65535 ]


#### Volvemos a guardar el dataset preprocesado

In [18]:
df.to_parquet(preprocessed_output, engine="pyarrow", index=False)